# 🎙 Discord Voice Bot — Google Colab

Запуск голосового Discord-бота на бесплатном GPU (T4) в Google Colab.

**Что умеет бот:**
- 🗣 Слушает голос в Discord (STT: faster-whisper на GPU)
- 🤖 Отвечает через LLM (любой OpenAI-совместимый API)
- 🔊 Озвучивает ответы (TTS: Silero v4 на GPU)
- 🎵 Включает музыку с YouTube по запросу
- 🔍 Ищет в интернете (DuckDuckGo + LLM переформулировка)
- 🔗 Кидает ссылки в текстовый чат

---

### Инструкция:
1. **Runtime → Change runtime type → T4 GPU**
2. Заполни токены в ячейке **Настройки**
3. Запусти все ячейки по порядку (Ctrl+F9)
4. Бот автоматически зайдёт в голосовой канал (если указан `AUTO_JOIN_CHANNEL_ID`)

In [ ]:
#@title ⚙️ 1. Настройки (заполни свои токены)

#@markdown ### Обязательные токены
DISCORD_BOT_TOKEN = '' #@param {type:"string"}
OPENAI_API_KEY = '' #@param {type:"string"}
OPENAI_BASE_URL = 'https://api.polza.ai/api/v1' #@param {type:"string"}

#@markdown ---
#@markdown ### LLM
LLM_MODEL = 'gemini-2.5-flash-lite' #@param {type:"string"}
LLM_MAX_TOKENS = 150 #@param {type:"integer"}
LLM_TEMPERATURE = 0.9 #@param {type:"number"}

#@markdown ---
#@markdown ### Голос (STT / TTS)
STT_MODEL = 'large-v3-turbo' #@param ['large-v3-turbo', 'large-v3', 'medium', 'small', 'base']
STT_LANGUAGE = 'ru' #@param {type:"string"}
TTS_VOICE = 'xenia' #@param ['xenia', 'aidar', 'baya', 'kseniya', 'eugene']

#@markdown ---
#@markdown ### Бот
BOT_NAME = 'Андрей' #@param {type:"string"}
AUTO_JOIN_CHANNEL_ID = '' #@param {type:"string"}

assert DISCORD_BOT_TOKEN, "❌ Вставь DISCORD_BOT_TOKEN!"
assert OPENAI_API_KEY, "❌ Вставь OPENAI_API_KEY!"
print("✅ Настройки заполнены")

In [ ]:
#@title 📦 2. Установка системных зависимостей
%%bash
apt-get update -qq
apt-get install -y -qq ffmpeg portaudio19-dev python3-dev > /dev/null 2>&1
echo "✅ ffmpeg и portaudio установлены"

In [ ]:
#@title 📥 3. Клонирование репозитория + установка Python зависимостей
import os

# Клонируем или обновляем
if not os.path.exists('bot'):
    !git clone -b low-latency https://github.com/karkqm/discord-voice-bott.git bot
else:
    !cd bot && git pull origin low-latency

os.chdir('bot')
print(f"📂 Рабочая директория: {os.getcwd()}")

# Устанавливаем зависимости
!pip install -q \
    "py-cord[voice]==2.6.1" \
    "openai>=1.40.0" \
    "python-dotenv>=1.0.0" \
    "RealtimeSTT>=0.3.0" \
    "faster-whisper>=1.0.0" \
    "silero-tts>=0.0.5" \
    "omegaconf>=2.3.0" \
    "PyNaCl>=1.5.0" \
    "Pillow>=10.0.0" \
    "aiohttp>=3.9.0" \
    "numpy>=1.24.0" \
    "duckduckgo-search>=6.0.0" \
    "yt-dlp>=2024.0.0" \
    "PyAudio"

print("✅ Все зависимости установлены")

In [ ]:
#@title 🔍 4. Проверка GPU
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f"✅ GPU: {gpu} ({vram:.1f} GB VRAM)")
    print(f"   CUDA: {torch.version.cuda}")
else:
    print("❌ GPU не найден! Runtime → Change runtime type → T4 GPU")

In [ ]:
#@title 📝 5. Создание .env файла
env_content = f"""DISCORD_BOT_TOKEN={DISCORD_BOT_TOKEN}
OPENAI_API_KEY={OPENAI_API_KEY}
OPENAI_BASE_URL={OPENAI_BASE_URL}

LLM_MODEL={LLM_MODEL}
LLM_MAX_TOKENS={LLM_MAX_TOKENS}
LLM_TEMPERATURE={LLM_TEMPERATURE}
IS_LOCAL_LLM=false

STT_BACKEND=realtime
STT_MODEL={STT_MODEL}
STT_LANGUAGE={STT_LANGUAGE}

TTS_ENGINE=silero
TTS_VOICE={TTS_VOICE}

SCREEN_CAPTURE_INTERVAL=5
SCREEN_CAPTURE_ENABLED=false

AUTO_JOIN_CHANNEL_ID={AUTO_JOIN_CHANNEL_ID}

BOT_NAME={BOT_NAME}
BOT_ALIASES=бот,алекс,андрей,слышь
BARGE_IN_SENSITIVITY=0.5
"""

with open('.env', 'w') as f:
    f.write(env_content)

print("✅ .env создан")
print(f"   LLM: {LLM_MODEL} @ {OPENAI_BASE_URL}")
print(f"   STT: realtime / {STT_MODEL}")
print(f"   TTS: silero / {TTS_VOICE}")
print(f"   Бот: {BOT_NAME}")
if AUTO_JOIN_CHANNEL_ID:
    print(f"   Авто-вход: канал {AUTO_JOIN_CHANNEL_ID}")
else:
    print("   Авто-вход: выключен (используй !join в Discord)")

In [ ]:
#@title 🚀 6. Запуск бота
!python bot.py